# Azure Face Service Quickstart
Este notebook demonstra como interagir com o serviço de Reconhecimento Facial do Azure AI utilizando o SDK oficial para Python.

## Pré-requisitos
Antes de executar as células de código:
1. Tenha uma instância do **Azure AI Services - Face** provisionada.
2. Defina as variáveis de ambiente `FACE_APIKEY` e `FACE_ENDPOINT` com a chave e o endpoint do recurso.
3. Opcionalmente, utilize um arquivo `.env` ou mecanismo equivalente para persistir essas variáveis.

## Instalação de dependências

In [ ]:

# Execute esta célula apenas uma vez por ambiente de execução.
# O parâmetro --quiet reduz a verbosidade da instalação.
!pip install --quiet azure-ai-vision-face


## Importações e configuração

In [ ]:

import os
import time
import uuid

from azure.core.credentials import AzureKeyCredential
from azure.ai.vision.face import FaceAdministrationClient, FaceClient
from azure.ai.vision.face.models import (
    FaceAttributeTypeRecognition04,
    FaceDetectionModel,
    FaceRecognitionModel,
    QualityForRecognition,
)

face_api_key = os.getenv("FACE_APIKEY")
face_endpoint = os.getenv("FACE_ENDPOINT")

if not face_api_key or not face_endpoint:
    raise ValueError("Defina as variáveis de ambiente FACE_APIKEY e FACE_ENDPOINT antes de continuar.")

LARGE_PERSON_GROUP_ID = str(uuid.uuid4())
print(f"Person group: {LARGE_PERSON_GROUP_ID}")


## Conjunto de imagens de exemplo
Os arquivos são obtidos do repositório público de exemplos da Microsoft.

In [ ]:

woman_images = [
    "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/Family1-Mom1.jpg",
    "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/Family1-Mom2.jpg",
]
man_images = [
    "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/Family1-Dad1.jpg",
    "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/Family1-Dad2.jpg",
]
child_images = [
    "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/Family1-Son1.jpg",
    "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/Family1-Son2.jpg",
]
test_image = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/Face/images/identification1.jpg"


## Criação, treinamento e teste do Large Person Group

In [ ]:

with FaceAdministrationClient(endpoint=face_endpoint, credential=AzureKeyCredential(face_api_key)) as face_admin_client,          FaceClient(endpoint=face_endpoint, credential=AzureKeyCredential(face_api_key)) as face_client:
    # Criação do grupo de pessoas
    face_admin_client.large_person_group.create(
        large_person_group_id=LARGE_PERSON_GROUP_ID,
        name=LARGE_PERSON_GROUP_ID,
        recognition_model=FaceRecognitionModel.RECOGNITION04,
    )

    woman = face_admin_client.large_person_group.create_person(
        large_person_group_id=LARGE_PERSON_GROUP_ID,
        name="Woman",
    )
    man = face_admin_client.large_person_group.create_person(
        large_person_group_id=LARGE_PERSON_GROUP_ID,
        name="Man",
    )
    child = face_admin_client.large_person_group.create_person(
        large_person_group_id=LARGE_PERSON_GROUP_ID,
        name="Child",
    )

    def _add_faces(image_urls, person_id):
        for image in image_urls:
            sufficient_quality = True
            detected_faces = face_client.detect_from_url(
                url=image,
                detection_model=FaceDetectionModel.DETECTION03,
                recognition_model=FaceRecognitionModel.RECOGNITION04,
                return_face_id=True,
                return_face_attributes=[FaceAttributeTypeRecognition04.QUALITY_FOR_RECOGNITION],
            )

            for face in detected_faces:
                if face.face_attributes.quality_for_recognition != QualityForRecognition.HIGH:
                    sufficient_quality = False
                    break

            if not sufficient_quality or len(detected_faces) != 1:
                continue

            face_admin_client.large_person_group.add_face_from_url(
                large_person_group_id=LARGE_PERSON_GROUP_ID,
                person_id=person_id,
                url=image,
                detection_model=FaceDetectionModel.DETECTION03,
            )
            print(f"Face {detected_faces[0].face_id} adicionada ao cadastro {person_id}")

    _add_faces(woman_images, woman.person_id)
    _add_faces(man_images, man.person_id)
    _add_faces(child_images, child.person_id)

    # Treinamento
    print(f"Treinando o grupo {LARGE_PERSON_GROUP_ID}...")
    poller = face_admin_client.large_person_group.begin_train(
        large_person_group_id=LARGE_PERSON_GROUP_ID,
        polling_interval=5,
    )
    poller.wait()
    print("Treinamento concluído.")

    print("Aguardando 60 segundos para evitar limites de taxa...")
    time.sleep(60)

    faces = face_client.detect_from_url(
        url=test_image,
        detection_model=FaceDetectionModel.DETECTION03,
        recognition_model=FaceRecognitionModel.RECOGNITION04,
        return_face_id=True,
        return_face_attributes=[FaceAttributeTypeRecognition04.QUALITY_FOR_RECOGNITION],
    )
    face_ids = [
        face.face_id
        for face in faces
        if face.face_attributes.quality_for_recognition != QualityForRecognition.LOW
    ]

    identify_results = face_client.identify_from_large_person_group(
        face_ids=face_ids,
        large_person_group_id=LARGE_PERSON_GROUP_ID,
    )

    for identify_result in identify_results:
        if identify_result.candidates:
            top_candidate = identify_result.candidates[0]
            print(
                f"Face {identify_result.face_id} corresponde a {top_candidate.person_id} com confiança "
                f"{top_candidate.confidence:.2f}"
            )
            verify_result = face_client.verify_from_large_person_group(
                face_id=identify_result.face_id,
                large_person_group_id=LARGE_PERSON_GROUP_ID,
                person_id=top_candidate.person_id,
            )
            print(
                f"Verificação: {'igual' if verify_result.is_identical else 'diferente'} | "
                f"Confiança: {verify_result.confidence:.2f}"
            )
        else:
            print(f"Nenhuma pessoa identificada para o face ID {identify_result.face_id}.")

    face_admin_client.large_person_group.delete(LARGE_PERSON_GROUP_ID)
    print(f"Grupo {LARGE_PERSON_GROUP_ID} excluído.")
